# Magnetic Resonance Imaging II
 
## Exercise 3
 
---
 
**Prof. Dr. Markus Zimmermann**
 
**Mohamed Wassim Bensalem**
 
---

TODO

Implement your own EPI sequence.

Investigate the effect of acceleration.

Start with R=1. Implement your solution so that R can be set flexibly. Compare R=1 vs R=2 in EPI.

## Software Imports
 
All required libraries and modules are imported here.

In [ ]:
%matplotlib widget

import sys
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
from tqdm import tqdm

sys.path.insert(0, "/home/jovyan/mri/src")
sys.path.insert(0, "/home/jovyan/shared-folder/Magnetresonanztomographie-2/mri/src")

from mri.conopt.linops import LinearOperator
from mri.parim.analytic import cyclic_head_coil

## System and Sequence Parameters
 
The parameters are identical to Exercise 1, with one addition: `n_channels` defines the number of receiver coils in the array. `n_x` and `n_y` define the full k-space matrix size — note that the accelerated acquisition will only acquire `n_y // 2` of the $n_y$ phase-encoding lines, but the reconstruction target remains an $n_x \times n_y$ image.

In [ ]:
seq_filename = "epi.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128
n_channels = 8
# Acceleration factor
R = 2

case = "EPI"
flip_angle_deg = 90
te = 25e-3

torch.set_default_device("cpu")

## System and Sequence Object

In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)

## Linear Operators

TODO

The linear operators are identical to Exercise 2. They are provided here so you can focus on the new content.

In [ ]:
class FourierTransformOperator(LinearOperator):

    def __init__(
        self,
        data_shape: Tuple[int, ...],
        axes: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):

        self.forward_shape = data_shape
        self.adjoint_shape = data_shape
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.internal_dtype = torch.complex64
        self.dtype = torch.float32
        self.device = device
        self.axes = axes

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        x = torch.fft.ifftshift(x, dim=self.axes)
        x = torch.fft.fftn(x, dim=self.axes, norm="ortho")
        x = torch.fft.fftshift(x, dim=self.axes)
        return x.view(self.dtype).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        x = torch.fft.ifftshift(x, dim=self.axes)
        x = torch.fft.ifftn(x, dim=self.axes, norm="ortho")
        x = torch.fft.fftshift(x, dim=self.axes)
        return x.view(self.dtype).ravel()


class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nX, nY = data_shape
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities
        self.device = device
        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.forward_shape = (1, nX, nY)
        self.adjoint_shape = (nC, nX, nY)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return y.view(torch.float32).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return y.view(torch.float32).ravel()


class DiagonalOperator(LinearOperator):

    def __init__(
        self,
        diagonal_tensor: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        self.diagonal_tensor = diagonal_tensor
        self.device = device
        self.internal_dtype = torch.complex64
        self.dtype = torch.float32
        self.forward_shape = data_shape
        self.adjoint_shape = data_shape
        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        y = self.diagonal_tensor * x
        return y.view(torch.float32).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        y = torch.conj(self.diagonal_tensor) * x
        return y.view(torch.float32).ravel()

## Task 1: Sequence Components

TODO

In [ ]:
# Your solution here


delta_kx = 1 / fov_x
delta_ky = 1 / fov_y
raster_eps = 1e-12
flat_time = 1.6e-3


rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

gx = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=flat_time,
    system=system,
)

gx_rev = pp.make_trapezoid(
    channel="x",
    flat_area=-n_x * delta_kx,
    flat_time=flat_time,
    system=system,
)

gx_pre = pp.make_trapezoid(
    channel="x",
    area=-gx.area / 2,
    system=system,
)

gy_pre = pp.make_trapezoid(
    channel="y",
    area=-n_y * delta_ky / 2,
    system=system,
)

gy_blip = pp.make_trapezoid(
    channel="y",
    area=R * delta_ky,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=flat_time,
    delay=gx.rise_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * gx.area,
    system=system,
)

te_min = (
    pp.calc_duration(rf)
    - pp.calc_rf_center(rf)[0]
    + pp.calc_duration(gx_pre)
    + n_y / R * pp.calc_duration(gx)
)

te_delay = (
    np.ceil((te - te_min - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)

if te_delay <= 0:
    print(f"Echo time {te} shorter than minimal echo time {te_min} -> Set echo time to {te_min}")
    te_delay = 0
    te = te_min

## Task 2: EPI Sequence Assembly

TODO

In [ ]:
# Your solution here


seq.add_block(rf)

seq.add_block(gx_pre, gy_pre)

seq.add_block(pp.make_delay(te_delay))

for index in range(n_y // (2 * R)):

    seq.add_block(gx, adc)

    seq.add_block(gy_blip)

    seq.add_block(gx_rev, adc)

    seq.add_block(gy_blip)

seq.add_block(gx_spoiler)

## Verification and Export

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot()

seq.write(seq_filename, v141_compat=True)

## Phantom and Multi-Channel Simulation

TODO

In [ ]:
phantom = mr0.VoxelGridPhantom.brainweb("subject05.npz")

phantom = phantom.interpolate(n_x, n_y, 32).slices([16])
phantom.size[2]=0.08

coil_sensitivities_recon = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
).to(torch.complex64)
coil_sensitivities_recon = coil_sensitivities_recon[:, :, :, 0].flip((1, 2))

coil_sensitivities_sim = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
    fov_scaling=(phantom.size[0] / fov_x, phantom.size[1] / fov_y, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim

plt.figure()
phantom.plot()

mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

## Task 3: Coil Images

TODO

In [ ]:
# Your solution here


data = signal.clone()
data = data.reshape((n_y // R, n_x, n_channels)).permute((2, 1, 0))

data_full = torch.zeros((n_channels, n_x, n_y), dtype=data.dtype)
data_full[:, :, ::R] = data

data_full[:, :, R :: 2 * R] = torch.flip(data_full[:, :, R :: 2 * R], (1,))

coil_images = torch.fft.fftshift(
    torch.fft.ifftn(
        torch.fft.ifftshift(data_full, dim=(1, 2)),
        dim=(1, 2),
    ),
    dim=(1, 2),
)


plt.figure()
for i in range(n_channels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(torch.abs(coil_images[i]).numpy().T, cmap="gray")
    plt.title(f"Channel {i + 1}")
plt.suptitle("Coil images")
plt.tight_layout()
plt.show()

plt.figure()
for i in range(n_channels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(torch.abs(coil_sensitivities_recon[i]).numpy().T, cmap="viridis")
    plt.title(f"Channel {i + 1}")
plt.suptitle("Coil sensitivity maps")
plt.tight_layout()
plt.show()

## Task 4: Compose the Forward Operator and Compute Regridded Image

TODO

In [ ]:
# Your solution here


mask = torch.zeros((n_channels, n_x, n_y), dtype=torch.complex64)
mask[:, :, ::R] = 1

F = FourierTransformOperator((n_channels, n_x, n_y), (1, 2))
C = ChannelOperator(coil_sensitivities_recon, (n_x, n_y))
M = DiagonalOperator(mask, (n_channels, n_x, n_y))

A = M @ F @ C

AHd = A.H @ data_full.view(torch.float32).ravel()

AHd_image = AHd.view(torch.complex64).reshape((n_x, n_y))

plt.figure()
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(
    f"$\\mathbf{{A}}^\\mathrm{{H}} \\mathbf{{d}}$ — Zero-filled Reconstruction ({case})"
)
plt.show()

## Task 5: Iterative Reconstruction with Conjugate Gradient

TODO

In [ ]:
def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(range(maxiter)):

        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

TODO

In [ ]:
# Your solution here


Q = A.H @ A

b = AHd
x = torch.zeros(n_x * n_y, dtype=torch.complex64).view(torch.float32)

x_recon = conjugate_gradient(Q, b, x, maxiter=30)

x_recon_image = x_recon.view(torch.complex64).reshape((n_x, n_y))

plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(f"Zero-filled ({case})")
plt.subplot(1, 2, 2)
plt.imshow(torch.abs(x_recon_image).numpy().T, cmap="gray")
plt.title(f"CG Reconstruction ({case})")
plt.tight_layout()
plt.show()